# Projeto 1: Limpeza dataset ecommerce

## Importação das bibliotecas


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Importação dos dados

In [2]:

importacao=pd.read_csv('vendas_ecommerce_dados_brutos.csv')
df=importacao.copy()

FileNotFoundError: [Errno 2] No such file or directory: 'vendas_ecommerce_dados_brutos.csv'

## Entendendo os dados


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 504 entries, 0 to 503
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_pedido          504 non-null    int64 
 1   data_pedido        501 non-null    object
 2   id_cliente         501 non-null    object
 3   nome_cliente       481 non-null    object
 4   email              483 non-null    object
 5   telefone           444 non-null    object
 6   idade              481 non-null    object
 7   cidade             486 non-null    object
 8   estado             492 non-null    object
 9   categoria_produto  500 non-null    object
 10  produto            502 non-null    object
 11  quantidade         490 non-null    object
 12  preco_unitario     490 non-null    object
 13  valor_total        472 non-null    object
 14  forma_pagamento    493 non-null    object
 15  status_pedido      501 non-null    object
 16  avaliacao_cliente  418 non-null    object
dt

In [ ]:
df.describe()

,id_pedido
count,504.000000
mean,1243.095238
std,140.767305
min,1001.000000
25%,1122.750000
50%,1241.500000
75%,1361.250000
max,1583.000000


## Tratando colunas


In [ ]:
#Tratando colunas numéricas float

#Limpando as colunas e mantendo somente os dados numéricos e os separadores de decimais e de milhar
for col in ['valor_total','preco_unitario']:
    df[col]=df[col].astype('str').str.replace(r'[^\d\.\,]','',regex=True).str.strip()

#lidando com as colunas numéricas float despadronizadas, algumas usavam ',' como separador de decimal e outras usavam 
# '.' oq dificultava em criar uma solução geral para a coluna 
for col in ['valor_total','preco_unitario']:
    virgula= df[col].str.contains(r',\d{2}$',regex=True)
    df.loc[virgula,col] = (df.loc[virgula,col].str.replace('.','')
                            .str.replace(',','.')
                            .str.strip())
    
#filtro=df['preco_unitario'].astype('str').str.contains(r'^\s$',regex=True)
#df.loc[filtro,'preco_unitario'] = np.nan   # não funcionou


for col in ['valor_total','preco_unitario']:
    df.loc[df[col]=='', col] = np.nan
    df[col]=df[col].astype('float')


In [ ]:
#colunas numericas int

# essas linhas servem para lidar com os caracteres que não são dígitos
df['avaliacao_cliente']=df['avaliacao_cliente'].str.replace(r'\D','',regex=True).str.strip()
df['quantidade']=df['quantidade'].str.replace(r'\D','',regex=True).str.strip()
df['quantidade'].value_counts()


In [ ]:
#trocando os espaços por nulos para poder criar colunas numéricas e imputar valores aos nulos
for col in ['quantidade','avaliacao_cliente']:
   # df[df[col]==''] = np.nan  #errado, isso apaga a linha toda, tem que especificar a coluna desejada
   df.loc[df[col]=='',col] = np.nan

# estava dando erro ao tentar converter diretamente para int por conta dos nulos, coloquei para float somente para realizar operações de mediana e tratar os nulos
df['quantidade']=df['quantidade'].astype('float')
df['avaliacao_cliente']=df['avaliacao_cliente'].astype('float')

#imputando a mediana aos nulos
for col in ['quantidade','valor_total','preco_unitario','avaliacao_cliente']:
   df[col]=df[col].fillna((df[col].median(skipna=True)))

#conversão para int
df['quantidade']=df['quantidade'].astype('Int64')
df['avaliacao_cliente']=df['avaliacao_cliente'].astype('Int64')

In [ ]:

df['id_cliente']=df['id_cliente'].astype('str').str.replace(r'\D','',regex=True).str.strip()
df.loc[df['id_cliente']=='','id_cliente'] = np.nan

# Exclusão das linhas com id nulo. 
# mesmo que estivesse com todas as outras colunas preenchidas sem id não é possível diferenciar/localizar entre os cadastros
df=df.dropna(subset='id_cliente')
df['id_pedido']=df['id_pedido'].astype('int')
df['id_cliente']=df['id_cliente'].astype('int')

In [ ]:
#retirando não dígitos
df['idade']=df['idade'].str.replace(r'\D','',regex=True)
df['idade']=df['idade'].str.strip()
df.loc[df['idade']=='','idade'] = np.nan

# O int padrão python/numpy não aceita nulos, diferente do Int do pandas (o float padrão aceita também).
df['idade']=df['idade'].astype('Int64')


In [ ]:
#imputando nulos para a coluna de idade
#a condição normalmente iria depender das regras de negócio
df.loc[df['idade']<18,'idade'] = df['idade'].quantile(0.25)
df['idade']=df['idade'].fillna((df['idade'].median()))
df.loc[df['idade']>80,'idade'] = df['idade'].quantile(0.75)


In [ ]:
#trocando os separadores das datas
df['data_pedido']=df['data_pedido'].str.replace('-','/')

#formatando a coluna para o tipo data
df['data_pedido']=pd.to_datetime(df['data_pedido'],format='mixed')

In [ ]:
#lidando com caracteres indesejados 
df['nome_cliente']=df['nome_cliente'].str.replace(r'[^\w\s\.]','',regex=True).str.title().str.strip()
df.loc[df['nome_cliente']=='','nome_cliente'] = np.nan

df.info()


In [ ]:

df['telefone']=df['telefone'].str.replace(r'[^\d\-]','',regex=True).str.replace(r'^55','',regex=True)
df.loc[df['telefone']=='','telefone'] = np.nan

for col in ['forma_pagamento','status_pedido','categoria_produto','cidade']:
    df[col]=df[col].astype('str').str.strip().str.title()
    df[col]=df[col].str.replace(r'[^\w\s]','',regex=True)
    df.loc[df[col]=='',col] = np.nan
    
df['produto']=df['produto'].astype('str').str.strip().str.title()
df['produto']=df['produto'].str.replace(r'[^\w\s]','',regex=True)
df.loc[df['produto']=='','produto'] = np.nan

df.info()

In [ ]:
df['email']=df['email'].str.replace(r'[^\w\.\+\-\@]','',regex=True).str.lower()
df.loc[df['email']=='','email'] = np.nan

In [ ]:
#padronizando os registros de estado
df['estado'] = df['estado'].str.upper()

mapa_estados = {
    'MINAS GERAIS': 'MG',
    'CEARÁ': 'CE', 'CEARA': 'CE',
    'BAHIA': 'BA',
    'PERNAMBUCO': 'PE',
    'RIO GRANDE DO SUL': 'RS',
    'AMAZONAS': 'AM',
    'RIO DE JANEIRO': 'RJ',
    'PARANÁ': 'PR', 'PARANA': 'PR',
    'SANTA CATARINA': 'SC',
    'SÃO PAULO': 'SP', 'SAO PAULO': 'SP',
    'SERGIPE': 'SE',
    'DISTRITO FEDERAL': 'DF',
    'NÃO INFORMADO': np.nan, 'NAO INFORMADO': np.nan, '?': np.nan,
}

df['estado'] = df['estado'].replace(mapa_estados)
df

In [ ]:
df=df.fillna('Não Informado')

## Exportação


In [ ]:
df.to_csv('ecommerce_limpo.csv', index=False)